In [3]:
import pandas as pd
from pyecharts import options as opts
from pyecharts.charts import Pie

# 由于数据错误，行业列里面是省份名称，需要排除的省份列表
exclude_provinces = ["北京", "广东", "山东", "四川", "安徽", "江苏", "辽宁", "宁夏", 
                    "浙江", "湖北", "陕西", "重庆", "内蒙古", "天津", "云南", "上海"]

# 读取数据
df = pd.read_csv('Alldata.csv', encoding='utf-8')

# 数据清洗
# 删除关键字段为空的数据
df = df.dropna(subset=['市值', '行业'])

# 删除行业列中包含省份名称的数据行
province_mask = df['行业'].apply(lambda x: any(province in str(x) for province in exclude_provinces))
print(f"\n发现市值列包含省份的记录数: {province_mask.sum()}")
print("将被删除的示例数据:")
print(df[province_mask].head()[['公司全称', '市值']])

df = df[~province_mask].copy()
print(f"删除后剩余数据量: {len(df)}")

# 市值单位转换函数
def convert_market_value(value):
    try: 
        # 提取数字部分
        num_str = ''.join([c for c in str(value).split('：')[1] if c.isdigit() or c == '.'])
        num = float(num_str)
        
        if '亿' in str(value):
            return num
        elif '万' in str(value):
            return num / 10000
        elif '美元' in str(value):
            return num * 7 / 10000 if '万' in str(value) else num * 7
        return num  # 默认按亿处理
    except:
        print(f"无法解析的市值值: {value}")
        return None

df['市值_亿'] = df['市值'].apply(convert_market_value)
df = df.dropna(subset=['市值_亿'])

# 按行业统计，使用清洗后的行业列
industry_data = (
    df.groupby('行业', as_index=False)['市值_亿']
    .sum()
    .sort_values('市值_亿', ascending=False)
)

# 绘制饼图
data_pair = [(row['行业'], round(row['市值_亿'], 2)) for _, row in industry_data.iterrows()]

pie = (
    Pie(init_opts=opts.InitOpts(width="1000px", height="600px"))
    .add(
        series_name="市值(亿元)",
        data_pair=data_pair,
        radius=["0%", "65%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(
            formatter="{b}|{c}亿|{d}%",
            font_size=12,
            rich={
                "b": {"fontSize": 14, "lineHeight": 33},
                "per": {"color": "#eee", "backgroundColor": "#334455"}
            }
        )
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="各行业市值占比分析",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=20)
        ),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_right="1%",
            pos_top="15%",
            item_width=40,
            item_height=20
        ),
        tooltip_opts=opts.TooltipOpts(
            trigger="item",
            formatter="{a}<br/>{b}: {c}亿元 ({d}%)"
        )
    )
    .set_series_opts(
        tooltip_opts=opts.TooltipOpts(trigger="item"),
        label_opts=opts.LabelOpts(
            position="outside",
            formatter="{b}: {d}%",
        )
    )
)

# 保存结果
output_file = "industry_market_share_final.html"
pie.render(output_file)
print(f"\n图表已生成：{output_file}")
print("\n最终行业分布：")
print(industry_data)


发现市值列包含省份的记录数: 64
将被删除的示例数据:
                 公司全称          市值
348    广东智通人才连锁股份有限公司    市值：2.60亿
385   随身云（南京）信息技术有限公司  市值：8280.0万
816      北京明弘科贸有限责任公司  市值：8000.0万
1290    上海美士达商务咨询有限公司    市值：10.0亿
1293   高联酒店管理（中国）有限公司  市值：7527.9万
删除后剩余数据量: 10307

图表已生成：industry_market_share_final.html

最终行业分布：
              行业           市值_亿
12            金融  163920.720180
0      IT/互联网/通信   98243.535712
11      能源/化工/矿业   38904.575303
8    机械/制造/军工/贸易   32331.329395
9        汽车/交通出行   21072.107122
5      建设工程/房产服务   19270.642158
4   广告/商务服务/文化传媒   17532.755953
3           医疗健康   12772.097207
13      零售/消费/食品   10517.234334
10      物流/仓储/运输    3643.553992
1             其它    1378.935440
7          旅游/酒店    1329.280391
2           农林牧渔     992.655580
6             教育     984.016781
